## Q3. Where should lending criteria be tightened or relaxed?
Purpose: a risk-tier table of trading-company borrower profiles (age band × numbered/named × ABN held), validated with a capture (lift) table, plus a red-flag override.
Metric: each profile's yearly deregistration probability at its current age, and its multiple of the trading-company average rate.
Significance: 95% interval on a rate ≈ rate × e^(±1.96/√d), d = deregistrations. A profile is Elevated/High only if the LOWER end of its interval is above the cut-off; Low only if the UPPER end is below 0.8.
Tiers (illustrative policy defaults): Low < 0.8× · Standard 0.8–1.25× · Elevated > 1.25× · High > 2× · Insufficient data (< 100 company-years or < 10 deregistrations).
Override: companies currently in strike-off (SOFF) or external administration (EXAD) → Decline / refer.
Tables: fct_exposure, fct_company_snapshot, dim_company_classification, dim_company, dim_age_band.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW q3_profile_risk AS
-- Step 1: trading-company exposure with the three profile attributes
WITH trading AS (
    SELECT
        f.age_band_key,
        f.exposure_years,
        f.deregistrations,
        CASE WHEN c.is_acn_named THEN 'Numbered' ELSE 'Named'  END AS name_type,
        CASE WHEN c.has_abn      THEN 'Has ABN'  ELSE 'No ABN' END AS abn_held
    FROM workspace.sme_risk_gold.fct_exposure f
    JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key = k.classification_key
    JOIN workspace.sme_risk_gold.dim_company c                ON f.company_key        = c.company_key
    WHERE k.borrower_segment = 'Trading company'
),

-- Step 2: benchmark = average deregistration rate of all trading companies
benchmark AS (
    SELECT SUM(deregistrations) / SUM(exposure_years) AS bench_rate
    FROM trading
),

-- Step 3: exposure and deregistrations per profile
profiles AS (
    SELECT age_band_key, name_type, abn_held,
           SUM(exposure_years)  AS exposure_years,
           SUM(deregistrations) AS deregistrations
    FROM trading
    GROUP BY age_band_key, name_type, abn_held
),

-- Step 4: how many trading companies currently sit in each profile (active, not distressed)
active_now AS (
    SELECT
        s.age_band_key,
        CASE WHEN c.is_acn_named THEN 'Numbered' ELSE 'Named'  END AS name_type,
        CASE WHEN c.has_abn      THEN 'Has ABN'  ELSE 'No ABN' END AS abn_held,
        COUNT(*) AS active_companies
    FROM workspace.sme_risk_gold.fct_company_snapshot s
    JOIN workspace.sme_risk_gold.dim_company_classification k ON s.classification_key = k.classification_key
    JOIN workspace.sme_risk_gold.dim_company c                ON s.company_key        = c.company_key
    WHERE k.borrower_segment = 'Trading company'
      AND s.is_active = 1
      AND s.is_distressed = 0
    GROUP BY 1, 2, 3
),

-- Step 5: rate, probability, multiple and its 95% interval
measured AS (
    SELECT
        p.*,
        a.age_band_label,
        p.deregistrations / p.exposure_years                                         AS dereg_rate,
        1 - EXP(-p.deregistrations / p.exposure_years)                               AS yearly_prob,
        (p.deregistrations / p.exposure_years) / b.bench_rate                        AS multiple,
        CASE WHEN p.deregistrations > 0
             THEN (p.deregistrations / p.exposure_years) * EXP(-1.96 / SQRT(p.deregistrations)) / b.bench_rate END AS multiple_low,
        CASE WHEN p.deregistrations > 0
             THEN (p.deregistrations / p.exposure_years) * EXP( 1.96 / SQRT(p.deregistrations)) / b.bench_rate END AS multiple_high,
        COALESCE(n.active_companies, 0)                                              AS active_companies
    FROM profiles p
    CROSS JOIN benchmark b
    JOIN workspace.sme_risk_gold.dim_age_band a ON p.age_band_key = a.age_band_key
    LEFT JOIN active_now n
        ON  p.age_band_key = n.age_band_key
        AND p.name_type    = n.name_type
        AND p.abn_held     = n.abn_held
)

-- Step 6: assign the tier
SELECT
    *,
    CASE
        WHEN exposure_years < 100 OR deregistrations < 10 THEN 'Insufficient data'
        WHEN multiple_low  > 2.00 THEN 'High'
        WHEN multiple_low  > 1.25 THEN 'Elevated'
        WHEN multiple_high < 0.80 THEN 'Low'
        ELSE 'Standard'
    END AS risk_tier,
    CASE
        WHEN exposure_years < 100 OR deregistrations < 10 THEN 0
        WHEN multiple_low  > 2.00 THEN 4
        WHEN multiple_low  > 1.25 THEN 3
        WHEN multiple_high < 0.80 THEN 1
        ELSE 2
    END AS tier_order
FROM measured;

In [0]:
%sql
-- Show the tier table: riskiest profiles first
SELECT
    risk_tier,
    age_band_label,
    name_type,
    abn_held,
    active_companies,
    ROUND(exposure_years, 0)       AS company_years,
    deregistrations,
    ROUND(100 * yearly_prob, 2)    AS yearly_dereg_prob_pct,
    ROUND(multiple, 2)             AS x_trading_average,
    ROUND(multiple_low, 2)         AS x_low_95,
    ROUND(multiple_high, 2)        AS x_high_95
FROM q3_profile_risk
ORDER BY tier_order DESC, multiple DESC;

In [0]:
SELECT
    risk_tier,
    COUNT(*)                                                                    AS profiles,
    SUM(active_companies)                                                       AS active_companies,
    ROUND(100.0 * SUM(active_companies) / SUM(SUM(active_companies)) OVER (), 1) AS pct_of_active_companies,
    ROUND(100.0 * SUM(exposure_years)   / SUM(SUM(exposure_years))   OVER (), 1) AS pct_of_company_years,
    ROUND(100.0 * SUM(deregistrations)  / SUM(SUM(deregistrations))  OVER (), 1) AS pct_of_deregistrations,
    ROUND((SUM(deregistrations) / SUM(SUM(deregistrations)) OVER ())
        / (SUM(exposure_years)  / SUM(SUM(exposure_years))  OVER ()), 2)        AS lift
FROM q3_profile_risk
GROUP BY risk_tier, tier_order
ORDER BY tier_order DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    s.status_code,
    s.status_label,
    COUNT(*)                                                    AS companies,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)          AS pct_of_red_flags
FROM workspace.sme_risk_gold.fct_company_snapshot f
JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key = k.classification_key
JOIN workspace.sme_risk_gold.dim_status s                 ON f.status_key         = s.status_key
WHERE k.borrower_segment = 'Trading company'
  AND f.is_distressed = 1
GROUP BY s.status_code, s.status_label
ORDER BY companies DESC;

In [0]:
%sql
-- Replace the ACN to look up any company (example: the most-renamed company)
SELECT
    c.acn,
    c.company_name,
    k.borrower_segment,
    st.status_label,
    a.age_band_label,
    CASE WHEN c.is_acn_named THEN 'Numbered' ELSE 'Named'  END AS name_type,
    CASE WHEN c.has_abn      THEN 'Has ABN'  ELSE 'No ABN' END AS abn_held,
    CASE
        WHEN f.is_distressed = 1                      THEN 'Decline / refer (red flag)'
        WHEN f.is_deregistered = 1                    THEN 'Deregistered'
        WHEN k.borrower_segment <> 'Trading company'  THEN 'Out of scope (not a trading company)'
        ELSE r.risk_tier
    END                                                        AS lending_tier,
    ROUND(100 * r.yearly_prob, 2)                              AS profile_yearly_dereg_prob_pct,
    ROUND(r.multiple, 2)                                       AS x_trading_average
FROM workspace.sme_risk_gold.dim_company c
JOIN workspace.sme_risk_gold.fct_company_snapshot f       ON c.company_key        = f.company_key
JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key = k.classification_key
JOIN workspace.sme_risk_gold.dim_status st                ON f.status_key         = st.status_key
JOIN workspace.sme_risk_gold.dim_age_band a               ON f.age_band_key       = a.age_band_key
LEFT JOIN q3_profile_risk r
    ON  r.age_band_key = f.age_band_key
    AND r.name_type    = CASE WHEN c.is_acn_named THEN 'Numbered' ELSE 'Named'  END
    AND r.abn_held     = CASE WHEN c.has_abn      THEN 'Has ABN'  ELSE 'No ABN' END
WHERE c.acn = '000386685';